In [ ]:
import torch 
from torch import nn
def net(x, y, z):
    return None
def output_head(y):
    return None
def Q_head(y):
    return None
y_init = None
z_init = None
N_supervision = None
train_dataloader = None
def input_embedding(x_input):
    return None
def latent_recursion(x, y, z, n):
    return None, None
def softmax_cross_entropy(pred, true):
    return None
def binary_cross_entropy(pred, true):
    return None
opt = None
def parameters_of(*modules):
    return []
def copy_parameters(*modules):
    return []
def target_generator(x_true, y_true, step):
    return None

### Original TRM code 

In [ ]:
def latent_recursion(x, y, z, n=6):
    for i in range(n): # latent reasoning
        z = net(x, y, z)
    y = net(y, z) # refine output answer
    return y, z

def deep_recursion(x, y, z, n=6, T=3):
    # recursing T−1 times to improve y and z (no gradients needed)
    with torch.no_grad():
        for j in range(T-1):
            y, z = latent_recursion(x, y, z, n)
    # recursing once to improve y and z
    y, z = latent_recursion(x, y, z, n)
    return (y.detach(), z.detach()), output_head(y), Q_head(y)

# Deep Supervision
for x_input, y_true in train_dataloader:
    y, z = y_init, z_init
    x = input_embedding(x_input) # x does not need to be instantiated at every step, since it is not updated during recursion
    for step in range(N_supervision):
        (y, z), y_hat, q_hat = deep_recursion(x, y, z)
        loss = softmax_cross_entropy(y_hat, y_true)
        loss += binary_cross_entropy(q_hat, (y_hat == y_true))
        loss.backward()
        opt.step()
        opt.zero_grad()
        if q_hat > 0: # early−stopping
            break

### TRM with EMA on weights

In [ ]:
def latent_recursion(x, y, z, n=6):
    for i in range(n): # latent reasoning
        z = net(x, y, z)
    y = net(y, z) # refine output answer
    return y, z

def deep_recursion(x, y, z, n=6, T=3):
    # recursing T−1 times to improve y and z (no gradients needed)
    with torch.no_grad():
        for j in range(T-1):
            y, z = latent_recursion(x, y, z, n)
    # recursing once to improve y and z
    y, z = latent_recursion(x, y, z, n)
    return (y.detach(), z.detach()), output_head(y), Q_head(y)

# before training 
model_params = parameters_of(net, input_embedding, output_head, Q_head)
ema_params = [p.detach().clone() for p in model_params]
beta = 0.999

# Deep Supervision
for x_input, y_true in train_dataloader:
    y, z = y_init, z_init
    x = input_embedding(x_input) # x does not need to be instantiated at every step, since it is not updated during recursion
    for step in range(N_supervision):
        (y, z), y_hat, q_hat = deep_recursion(x, y, z)
        loss = softmax_cross_entropy(y_hat, y_true)
        loss += binary_cross_entropy(q_hat, (y_hat == y_true))
        loss.backward()
        opt.step()
        opt.zero_grad()
        
        # EMA update: use persistent shadow weights
        model_params = parameters_of(net, input_embedding, output_head, Q_head)
        for ema_p, p in zip(ema_params, model_params):
            ema_p.data = beta * ema_p.data + (1.0 - beta) * p.data
            
        if q_hat > 0: # early−stopping
            break

### TRM with Target Generator (DIS)

In [ ]:
def latent_reasoning(x, y, z, n=2, T=3): 
    with torch.no_grad():
        for j in range(T-1): 
            for i in range(n): 
                z = net(x, y, z) 
            y = net(y, z) 
    for i in range(n): 
        z = net(x, y, z) 
    y = net(y, z) 
    return (y.detach(), z.detach()), output_head(y)

# Deep Improvement Supervision 
for x_input, y_true in train_dataloader: 
    y, z = y_init, z_init
    for step in range(N_supervision):
        x = input_embedding(x_input, step) 
        (y, z), y_hat = latent_reasoning(x, y, z)
        y_step = target_generator(x_true, y_true, step) # x_true should be x
        loss = softmax_cross_entropy(y_hat, y_step)
        loss.backward() 
        opt.step() 
        opt.zero_grad()

### TRM with learned EMA reweighter on latent states (RIM)

In [ ]:
z_gate = nn.Linear(z_dim, 1) 
y_gate = nn.Linear(y_dim, 1)  

def reweight(old, candidate, gate):
    alpha = torch.sigmoid(gate(candidate))
    return alpha * candidate + (1 - alpha) * old

def latent_recursion(x, y, z, n=6):
    for i in range(n):  # latent reasoning
        z_tilde = net(x, y, z) # Solver proposal
        z = reweight(z, z_tilde, z_gate)
    y_tilde = net(y, z)         # Generator proposal
    y = reweight(y, y_tilde, y_gate)
    return y, z

def deep_recursion(x, y, z, n=6, T=3):
    # recursing T−1 times to improve y and z (no gradients needed)
    with torch.no_grad():
        for j in range(T-1):
            y, z = latent_recursion(x, y, z, n)
    # recursing once to improve y and z
    y, z = latent_recursion(x, y, z, n)
    return (y.detach(), z.detach()), output_head(y), Q_head(y)

# Deep Supervision
for x_input, y_true in train_dataloader:
    y, z = y_init, z_init
    x = input_embedding(x_input) # x does not need to be instantiated at every step, since it is not updated during recursion
    for step in range(N_supervision):
        (y, z), y_hat, q_hat = deep_recursion(x, y, z)
        loss = softmax_cross_entropy(y_hat, y_true)
        loss += binary_cross_entropy(q_hat, (y_hat == y_true))
        loss.backward()
        opt.step()
        opt.zero_grad()
        if q_hat > 0: # early−stopping
            break

### TRM with Progressive Depth Curriculum (CGAR)

In [ ]:
def deep_recursion(x, y, z, n, T): 
    with torch.no_grad():
        for j in range(T-1): 
            for i in range(n): 
                z = net(x, y, z) 
            y = net(y, z) 
    for i in range(n): 
        z = net(x, y, z) 
    y = net(y, z) 
    return (y.detach(), z.detach()), output_head(y), Q_head(y)
        
def train_cgar (MAX_EPOCHS, PDC, lambda_decay, N_supervision) :
    Z_lambda = (1 - lambda_decay ** N_supervision) / (1 - lambda_decay)
    
    for epoch in range(1, MAX_EPOCHS + 1) :
        n , T = PDC(epoch / MAX_EPOCHS)
        
        for x_input, y_true in train_dataloader:
            y = input_embedding(x_input) # y becomes x embedded 
            z = z_init
            loss = 0.0
            for step in range (N_supervision) :
                (y, z), y_hat, q_hat = deep_recursion(x_input, y, z, n, T)
                weight = lambda_decay**step
                loss += weight * softmax_cross_entropy(y_hat, y_true)
                loss += binary_cross_entropy(q_hat, (y_hat == y_true)) 
                if max(q_hat) > 0: # or maybe 0.5
                    break
            loss = loss / Z_lambda
            loss.backward()
            opt.step()
            opt.zero_grad()
    return 

        

### TRM with Skip Connections (SRM)

In [ ]:
def latent_recursion(x, z, context, n=6):
    for i in range(n): # latent reasoning
        z = net(z, x, context)
    return z

def deep_recursion(x, z, context, trunc=2, n=6, T=3):
    # recursing T−trunc times to improve y and z (no gradients needed)
    with torch.no_grad():
        for j in range(T-trunc):
                for i in range(n): # latent reasoning
                    z = net(z, x, context)
    # recursing trunc times to improve z
    for j in range(trunc):
        for i in range(n): # latent reasoning
            z = net(z, x, context)
    return (z.detach()), output_head(y), Q_head(y)

def deep_recursion(x, z, trunc=1, n=6, T=3):
    for i in range(T): 
        context = f_new(z)
        
        with torch.no_grad():
            for j in range(n-trunc):
                z = net(z, x, context, n)
        
        for j in range(trunc):
            z = net(z, x, context, n)
        
        
    z = net(z, x, context, n)  
    return (z.detach()), output_head(y), Q_head(y) 
    
    
    # # recursing T−trunc times to improve y and z (no gradients needed)
    # with torch.no_grad():
    #     for j in range(T-trunc):
    #         z = net(z, x, context, n)
    # # recursing trunc times to improve z
    # for j in range(trunc):
    #     z = net(z, x, context, n)
    # return (z.detach()), output_head(y), Q_head(y)

# Deep Supervision
for x_input, y_true in train_dataloader:
    z = z_init
    x = input_embedding(x_input) # x does not need to be instantiated at every step, since it is not updated during recursion
    for step in range(N_supervision):
        (z), y_hat, q_hat = deep_recursion(z, x)
        loss = softmax_cross_entropy(y_hat, y_true)
        loss += binary_cross_entropy(q_hat, (y_hat == y_true))
        loss.backward()
        opt.step()
        opt.zero_grad()
        if q_hat > 0: # early−stopping
            break

In [ ]:

def deep_recursion(x, z, context, N=18, T=3):
    # recursing T−trunc times to improve z (no gradients needed)
    with torch.no_grad():
        for j in range(N-T):
            z = net(z, x, context)

    # recursing trunc times to improve z
    for j in range(T):
        z = net(z, x, context)
    return z.detach(), output_head(z), Q_head(z)

# Deep Supervision
for x_input, y_true in train_dataloader:
    z = z_init
    x = input_embedding(x_input) # x does not need to be instantiated at every step, since it is not updated during recursion
    for step in range(N_supervision):
        context = context_net(z) # e.g. z_start, net(z_start), or net_new(z_start)
        y, y_hat, q_hat = deep_recursion(x, z, context)
        loss = softmax_cross_entropy(y_hat, y_true)
        loss += binary_cross_entropy(q_hat, (y_hat == y_true))
        loss.backward()
        opt.step()
        opt.zero_grad()
        if q_hat > 0: # early−stopping
            break